# Access control in Unity Catalog


## 2. Privileges 

* Who can access what, using grants on securable objects.
* Applies to Catalogs, schemas, tables.

### Who can manage privileges?

Default Access:

* Databricks account admins, workspace admins, and metastore admins have default privileges for managing 
Unity Catalog.

Object Ownership:
* Every securable object in Unity Catalog has an owner.
* Object owners automatically have all privileges on their respective objects.
* Owners can grant the MANAGE privilege to other users.

Who Can Grant or Revoke Privileges?

* Object Owner: Owner of the specific securable object.
* Parent Catalog Owner: Owner of the catalog containing the object.
* Parent Schema Owner: Owner of the schema containing the object.
* MANAGE Privilege Holder: User with MANAGE on the specific object.
* Metastore Admin: Administrator of the metastore.
* Account Admin: Can grant privileges directly on a metastore.

### Show grants on an object

* Metastore admins, users with the READ METADATA or MANAGE privilege on the object, the owner of the object, or the owner of the catalog or schema that contains the object can see all grants on the object.
* If you do not have the above permissions, you can view only your own grants on the object.

In [0]:
%sql
  SHOW GRANTS ON SCHEMA bikesales.gold;
  SHOW GRANTS ON bikesales.gold.quarterly_sales_by_partner;

###Grant permissions on an object

* Metastore admin, the MANAGE privilege on the object, the owner of the object, or the owner of the catalog or schema that contains the object.

In [0]:
%sql
  GRANT CREATE TABLE ON SCHEMA main.default TO `finance-team`;
  GRANT USE SCHEMA ON SCHEMA main.default TO `finance-team`;
  GRANT USE CATALOG ON CATALOG main TO `finance-team`; 

### Revoke permissions on an object

Permissions required: Metastore admin, the MANAGE privilege on the object, the owner of the object, or the owner of the catalog or schema that contains the object.



In [0]:
%sql
REVOKE CREATE TABLE ON SCHEMA main.default FROM `finance-team`;

### Show grants on a metastore

Permissions required: Metastore admin or account admin.

In [0]:
%sql
SHOW GRANTS [principal] ON METASTORE

### Grant permissions on a metastore

Permissions required: Metastore admin or account admin.

In [0]:
GRANT <privilege-type> ON METASTORE TO <principal>;

### Revoke permissions on a metastore


Permissions required: Metastore admin or account admin.

In [0]:
%sql
REVOKE <privilege-type> ON METASTORE FROM <principal>;

## 2. Attribute-based access control in Unity Catalog

* ABAC (Attribute-Based Access Control) is an access-control mechanism in which access to data is determined by evaluating attributes/tags associated with data objects against predefined policies.
* ABAC uses two primary building blocks:
    * Governed Tags: Describe the characteristics of data.
    * Policies: Define what should happen when data matches particular attributes.
* The important concept is that the policy does not need to know the actual column name. It only needs to know the attribute assigned to that column. This is what makes ABAC scalable.

### How ABAC works

* A data steward classifies data using governed tags.
* A governance administrator creates policies.
* A user attempts to access a table.
* Unity Catalog evaluates applicable policies.
* The user receives data according to the applicable security rules.

![image_1795253895483.png](./image_1795253895483.png "image_1795253895483.png")

### Understanding the ABAC Architecture

![image_1795254163898.png](./image_1795254163898.png "image_1795254163898.png")

Imagine an organization has the following tables:
|Table|Column|Data classification|
|---|---|---|
|Customers|Customer_ID|Internal|
|Customers|Customer_Name|Internal|
|Customers|Email|PII|
|Customers|SSN|Highly Confidential|
|Employees|Employee_ID|Internal|
|Employees|Salary|Confidential|
|Employees|Email|PII|
|Orders|Order_ID|Internal|
|Orders|Customer_ID|Internal|


- Column Email has tag sensitivity = PII.
- A policy says mask all columns tagged as PII.
- A user queries the Customers table.
- Unity Catalog identifies the matching column and applies the masking function.

### Understanding Governed Tags

* Governed Tags are centrally managed key-value attributes that can be assigned to Unity Catalog securable objects.
* They are used to classify data and identify which objects should be governed by an ABAC policy.
* A tag consists of:
    - Key: Attribute name.
    - Value: Attribute value.
* Where can Governed Tags be applied?
    * Catalogs
    * Schemas
    * Tables
    * Columns
    * Models
    * Volumes
* Tag inheritance
    * Unity Catalog supports tag inheritance through the catalog and schema hierarchy.
    * The key rules:
        * Catalog tags can flow down to schemas and their descendants.
        * Schema tags can flow down to tables and their descendants.
        * Tags can be overridden at descendant levels, except at the column level.
        * Column tags do not inherit from their parent table. They must be applied directly.


![image_1795258049636.png](./image_1795258049636.png "image_1795258049636.png")

#### Creating and applying tags

* Governed tag definitions are created at the account level, through Catalog Explorer.
* You need the appropriate account-level CREATE permission.
* Syntax:
```
ALTER TABLE catalog_name.schema_name.table_name
ALTER COLUMN column_name
SET TAGS ('tag_key' = 'tag_value');
```

In [0]:
ALTER TABLE sales.customers.customer_details
ALTER COLUMN Email
UNSET TAGS ('pii');

### Understanding ABAC Policies


* A Policy is a set of rules that determines how Unity Catalog should handle data objects based on their attributes.
* A policy generally contains four important elements:
|Element|Purpose|Example|
|---|---|---|
|Scope|Where the policy applies|Catalog or Schema|
|Principals|Who the policy applies to|account users|
|Conditions|Which objects should match|has_tag('pii')|
|Action|What should happen|Mask a column|

#### Policy Scope – ON

* The ON clause defines the Unity Catalog object to which the policy is attached.
|Scope|Meaning|
|---|---|
|ON CATALOG|Applies to matching tables throughout the catalog|
|ON SCHEMA|Applies to matching tables within the schema|
|ON TABLE|Applies to the specified table|

#### Principals – TO and EXCEPT

* TO - Specifies who should be subject to the policy.
* EXCEPT - Specifies principals exempt from the policy.
* Example:
```
TO `account users`
EXCEPT `HR Admins`
```

#### Conditions

* WHEN identifies which object the tag is assigned to should be affected by the policy.
* Example:
```
WHEN has_tag('sensitivity')
WHEN has_tag_value('region', 'EU')
```
* MATCH COLUMNS identifies columns within the matching tags.
```
MATCH COLUMNS has_tag('pii') AS pii_col
MATCH COLUMNS has_tag_value('pii', 'ssn') AS ssn_col
```
* Databricks provides built-in functions to evaluate governed tags.
|Function|Purpose|Example|
|---|---|---|
|has_tag()|Checks whether a tag key exists|has_tag('pii')|
|has_tag_value()|Checks whether a tag has a specific value|has_tag_value('pii', 'ssn')|

#### Understanding UDFs in ABAC

* UDF stands for User-Defined Function.
* ABAC uses UDFs for two main policy types:
    * Row Filter Policies.
    * Column Mask Policies.

Example:
```
CREATE OR REPLACE FUNCTION mask_email(email STRING)
RETURNS STRING
RETURN '****';
```

##### Row Filter Policies

* Row filtering controls which records a user can see.


**Implementing Row Filtering**<br>
Step 1: Create a UDF:
```
CREATE OR REPLACE FUNCTION filter_region(
    row_region STRING,
    allowed_region STRING
)
RETURNS BOOLEAN
RETURN row_region = allowed_region;
```

Step 2: Create a Row Filter Policy:<br>
Conceptually, we want:
- Identify columns tagged as region.
- Pass their values to the UDF.
- Keep only records for which the UDF returns TRUE.
```
CREATE POLICY filter_us_region
ON SCHEMA sales.customers

ROW FILTER filter_region

TO `US Analysts`

FOR TABLES

MATCH COLUMNS has_tag('region') AS region_col

USING COLUMNS (region_col, 'US');
```

When a US Analyst executes:
```
SELECT * FROM sales.customers.orders;
```

|Order_ID|Customer|Region|Amount|
|---|---|---|---|
|101|John|US|1000|
|103|Bob|US|1500|




##### Column Mask Policies

* Column masking controls how values in particular columns are displayed to users.

**Implementing Column Masking**<br>
Step 1: Create the masking UDF
```
CREATE OR REPLACE FUNCTION mask_email(email STRING)
RETURNS STRING
RETURN '****';
```
Step 2: Create the Column Mask Policy
```
CREATE POLICY mask_pii_email
ON SCHEMA hr.employees

COLUMN MASK mask_email

TO `account users`

EXCEPT `HR Admins`

FOR TABLES

MATCH COLUMNS has_tag_value('pii', 'email') AS email_col

ON COLUMN email_col;
```


#### Understanding USING COLUMNS

* This is relevant when the requirement is such that it would require two columns to make a business decision.
* Example:
Requirement:
- If a customer has given consent, display the original email.
- Otherwise, mask the email.

How can we do this?

We need two columns to make the decision:
1. Email – the column to mask.
2. Has_Consent – the column that determines whether masking is necessary.

Step 1: Tag the columns<br>
```
ALTER TABLE customers.profiles
ALTER COLUMN Email
SET TAGS ('pii' = 'email');

ALTER TABLE customers.profiles
ALTER COLUMN Has_Consent
SET TAGS ('consent' = '');
```

Step 2: Create a UDF<br>
```
CREATE OR REPLACE FUNCTION mask_email_by_consent(
    email STRING,
    consent BOOLEAN
)
RETURNS STRING

RETURN CASE
    WHEN consent = TRUE THEN email
    ELSE '****'
END;
```

Step 3: Create the policy
```
CREATE POLICY conditional_email_mask
ON SCHEMA customers

COLUMN MASK mask_email_by_consent

TO `account users`

FOR TABLES

MATCH COLUMNS
    has_tag_value('pii', 'email') AS email_col,
    has_tag('consent') AS consent_col

ON COLUMN email_col

USING COLUMNS (consent_col);
```

#### Understanding FOR TABLES

* It specifies the type of objects the policy targets.

```
ON SCHEMA sales.customers
FOR TABLES
```

### Four Types of ABAC Policies

Row Filter Policy<br>
Controls which rows users can see.

Column Mask Policy<br>
Controls how column values are displayed.

GRANT Policy<br>
Dynamically grants Unity Catalog privileges when tag conditions match.

DENY Policy<br>
Explicitly denies a supported Unity Catalog privilege when the policy conditions match.

Grant Example:
```
CREATE POLICY grant_analyst_access
ON CATALOG sales

GRANT SELECT

TO `Sales Analysts`

FOR TABLES

WHEN has_tag_value('access_level', 'analyst');
```